# FreshCheck Colab Runner

ใช้ notebook นี้เป็นตัวกลางสำหรับ `GitHub + Colab + Google Drive`

Workflow:
1. Clone หรือ update repo จาก GitHub
2. Mount Google Drive
3. ติดตั้ง dependencies
4. รัน CLI `run_freshcheck.py`


In [1]:
#@title 1) Clone / Update repository from GitHub
REPO_URL = "https://github.com/techasit239/DADS7202_PigPicture.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/DADS7202_PigPicture"  #@param {type:"string"}

import os
import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    print(f"[INFO] Repo exists at {repo_path}. Pulling latest changes from {BRANCH}...")
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "origin", BRANCH], check=True)
else:
    print(f"[INFO] Cloning {REPO_URL} -> {repo_path}")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

%cd /content/DADS7202_PigPicture


[INFO] Cloning https://github.com/techasit239/DADS7202_PigPicture.git -> /content/DADS7202_PigPicture
/content/DADS7202_PigPicture


In [2]:
#@title 2) Mount Google Drive and define project paths
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')  #@param {type:"string"}
KAGGLE_DATA_DIR = DRIVE_ROOT / 'data' / 'kaggle_meat'
THAI_DATA_DIR = DRIVE_ROOT / 'data' / 'thai_retail'
CVAT_XML_PATH = DRIVE_ROOT / 'data' / 'thai_retail_cvat_annotations.xml'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

for folder in [DRIVE_ROOT, ARTIFACTS_DIR, ARTIFACTS_DIR / 'splits', ARTIFACTS_DIR / 'train', ARTIFACTS_DIR / 'eval', ARTIFACTS_DIR / 'predict', ARTIFACTS_DIR / 'phase2']:
    folder.mkdir(parents=True, exist_ok=True)

print('[OK] Drive mounted and directories prepared')
print('KAGGLE_DATA_DIR =', KAGGLE_DATA_DIR)
print('THAI_DATA_DIR   =', THAI_DATA_DIR)
print('CVAT_XML_PATH   =', CVAT_XML_PATH)
print('ARTIFACTS_DIR   =', ARTIFACTS_DIR)


Mounted at /content/drive
[OK] Drive mounted and directories prepared
KAGGLE_DATA_DIR = /content/drive/MyDrive/FreshCheck/data/kaggle_meat
THAI_DATA_DIR   = /content/drive/MyDrive/FreshCheck/data/thai_retail
CVAT_XML_PATH   = /content/drive/MyDrive/FreshCheck/data/thai_retail_cvat_annotations.xml
ARTIFACTS_DIR   = /content/drive/MyDrive/FreshCheck/artifacts


In [3]:
#@title 3) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


/content/DADS7202_PigPicture
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.7 MB/s eta 0:00:00


# Test case ใหม่ ตามสถานการณ์ต่าง ๆ ไม่รวม Dino

In [4]:
%cd /content/DADS7202_PigPicture
!git fetch origin
!git reset --hard origin/main
!python -m pip install -q -r requirements.txt

/content/DADS7202_PigPicture
HEAD is now at 740f215 Fix train summary metrics serialization


ถ้ายังไม่ได้สร้าง manifest/split ให้ใช้ชุดนี้ก่อน

In [5]:
!python run_freshcheck.py prepare-manifest \
  --data-dir "/content/drive/MyDrive/FreshCheck/data/kaggle_meat/Meat Freshness.v1-new-dataset.multiclass/train" \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/manifests \
  --layout kaggle \
  --filename source_train_manifest.csv

!python run_freshcheck.py prepare-manifest \
  --data-dir "/content/drive/MyDrive/FreshCheck/data/kaggle_meat/Meat Freshness.v1-new-dataset.multiclass/valid" \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/manifests \
  --layout kaggle \
  --filename source_holdout_manifest.csv

!python run_freshcheck.py prepare-manifest \
  --data-dir /content/drive/MyDrive/FreshCheck/data/new_test \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/manifests \
  --layout folder-labels \
  --filename target_manifest.csv

Saved manifest: /content/drive/MyDrive/FreshCheck/artifacts/manifests/source_train_manifest.csv
Saved manifest: /content/drive/MyDrive/FreshCheck/artifacts/manifests/source_holdout_manifest.csv
Saved manifest: /content/drive/MyDrive/FreshCheck/artifacts/manifests/target_manifest.csv


In [6]:
!python run_freshcheck.py split-manifest \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/manifests/target_manifest.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/target_split \
  --fit-ratio 0.7 \
  --fit-filename target_fit.csv \
  --holdout-filename target_holdout.csv \
  --seed 42

!python run_freshcheck.py split-manifest \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/target_split/target_fit.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/target_split \
  --fit-ratio 0.8 \
  --fit-filename target_train.csv \
  --holdout-filename target_val.csv \
  --seed 43

Saved fit split:     /content/drive/MyDrive/FreshCheck/artifacts/target_split/target_fit.csv
Saved holdout split: /content/drive/MyDrive/FreshCheck/artifacts/target_split/target_holdout.csv
Saved fit split:     /content/drive/MyDrive/FreshCheck/artifacts/target_split/target_train.csv
Saved holdout split: /content/drive/MyDrive/FreshCheck/artifacts/target_split/target_val.csv


สร้าง experiment manifests A/B/C/D อัตโนมัติด้วย CLI


In [ ]:
!python run_freshcheck.py prepare-experiments \
  --source-csv /content/drive/MyDrive/FreshCheck/artifacts/manifests/source_train_manifest.csv \
  --target-csv /content/drive/MyDrive/FreshCheck/artifacts/manifests/target_manifest.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto \
  --source-fit-ratio 0.8 \
  --target-fit-ratio 0.7 \
  --train-ratio-within-fit 0.8 \
  --seed 42


# วิธี A


In [ ]:
!python run_freshcheck.py train \
  --train-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/source_train.csv \
  --val-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/source_val.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_a_train \
  --models efficientnet_b0 convnext_tiny \
  --epochs 15


In [ ]:
!python run_freshcheck.py evaluate \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/target_holdout.csv \
  --checkpoint-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_a_train/checkpoints \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_a_eval_target \
  --models efficientnet_b0 convnext_tiny


# วิธี B


In [ ]:
!python run_freshcheck.py train \
  --train-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/exp_b_train.csv \
  --val-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/exp_b_val.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_b_train \
  --models efficientnet_b0 convnext_tiny \
  --epochs 15


In [ ]:
!python run_freshcheck.py evaluate \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/target_holdout.csv \
  --checkpoint-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_b_train/checkpoints \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_b_eval_target \
  --models efficientnet_b0 convnext_tiny


# วิธี C


In [ ]:
!python run_freshcheck.py train \
  --train-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/exp_c_train_rebalanced.csv \
  --val-csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/exp_c_val.csv \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_c_train \
  --models efficientnet_b0 convnext_tiny \
  --epochs 15


In [ ]:
!python run_freshcheck.py evaluate \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/target_holdout.csv \
  --checkpoint-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_c_train/checkpoints \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_c_eval_target \
  --models efficientnet_b0 convnext_tiny


# วิธี D


In [ ]:
!python run_freshcheck.py evaluate \
  --csv /content/drive/MyDrive/FreshCheck/artifacts/experiments_auto/source_holdout.csv \
  --checkpoint-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_c_train/checkpoints \
  --output-dir /content/drive/MyDrive/FreshCheck/artifacts/exp_d_eval_source \
  --models efficientnet_b0 convnext_tiny


# วิธี C+

C+ ใช้ชุด train/val เดียวกับ C แต่จูน hyperparameters แยกตามโมเดล เพื่อเพิ่มความแม่นบน target holdout โดยยังเช็ก source holdout ซ้ำอีกรอบ


In [ ]:
CPLUS_ROOT = '/content/drive/MyDrive/FreshCheck/artifacts'
EXP_C_TRAIN = f'{CPLUS_ROOT}/experiments_auto/exp_c_train_rebalanced.csv'
EXP_C_VAL = f'{CPLUS_ROOT}/experiments_auto/exp_c_val.csv'
TARGET_HOLDOUT = f'{CPLUS_ROOT}/experiments_auto/target_holdout.csv'
SOURCE_HOLDOUT = f'{CPLUS_ROOT}/experiments_auto/source_holdout.csv'

EFF_OUT = f'{CPLUS_ROOT}/exp_c_plus_efficientnet'
SWIN_OUT = f'{CPLUS_ROOT}/exp_c_plus_swin'
TARGET_EVAL_OUT = f'{CPLUS_ROOT}/exp_c_plus_eval_target'
SOURCE_EVAL_OUT = f'{CPLUS_ROOT}/exp_c_plus_eval_source'

print('train:', EXP_C_TRAIN)
print('val:  ', EXP_C_VAL)


In [ ]:
!python run_freshcheck.py train \
  --train-csv "$EXP_C_TRAIN" \
  --val-csv "$EXP_C_VAL" \
  --output-dir "$EFF_OUT" \
  --models efficientnet_b0 \
  --epochs 22 \
  --patience 6 \
  --lr 0.0003 \
  --weight-decay 0.005 \
  --dropout 0.25 \
  --img-size 224 \
  --batch-size 32 \
  --num-workers 2


In [ ]:
!python run_freshcheck.py train \
  --train-csv "$EXP_C_TRAIN" \
  --val-csv "$EXP_C_VAL" \
  --output-dir "$SWIN_OUT" \
  --models swin_t \
  --epochs 14 \
  --patience 4 \
  --lr 0.00008 \
  --weight-decay 0.01 \
  --dropout 0.30 \
  --img-size 224 \
  --batch-size 16 \
  --num-workers 2


In [ ]:
!python run_freshcheck.py evaluate \
  --csv "$TARGET_HOLDOUT" \
  --output-dir "$TARGET_EVAL_OUT" \
  --models efficientnet_b0 swin_t \
  --checkpoint-paths \
    efficientnet_b0="$EFF_OUT/checkpoints/efficientnet_b0_best.pt" \
    swin_t="$SWIN_OUT/checkpoints/swin_t_best.pt" \
  --batch-size 16 \
  --num-workers 2


In [ ]:
!python run_freshcheck.py evaluate \
  --csv "$SOURCE_HOLDOUT" \
  --output-dir "$SOURCE_EVAL_OUT" \
  --models efficientnet_b0 swin_t \
  --checkpoint-paths \
    efficientnet_b0="$EFF_OUT/checkpoints/efficientnet_b0_best.pt" \
    swin_t="$SWIN_OUT/checkpoints/swin_t_best.pt" \
  --batch-size 16 \
  --num-workers 2


ผลของ C+ จะถูกเก็บไว้ที่

- `/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_efficientnet`
- `/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_swin`
- `/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_eval_target`
- `/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_eval_source`

ไฟล์สำคัญที่ดูผลคือ `*_eval.json`, `*_confusion_matrix.csv`, `*_confusion_matrix.png`


In [ ]:
import json
from pathlib import Path

def read_eval(path):
    path = Path(path)
    if not path.exists():
        print(f'[MISSING] {path}')
        return None
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    results = data.get('results', {})
    print(f"\n=== {path.stem} ===")
    print(f"accuracy        : {results.get('accuracy', 0):.4f}")
    print(f"macro_precision : {results.get('macro_precision', 0):.4f}")
    print(f"macro_recall    : {results.get('macro_recall', 0):.4f}")
    print(f"macro_f1        : {results.get('macro_f1', 0):.4f}")
    print(f"loss            : {results.get('loss', 0):.4f}")
    cm = data.get('confusion_matrix', {})
    print('confusion labels:', cm.get('labels'))
    print('confusion matrix:')
    for row in cm.get('matrix', []):
        print(row)
    extra = data.get('extra', {})
    files = extra.get('confusion_matrix_files', {})
    if files:
        print('csv:', files.get('csv'))
        print('png:', files.get('png'))
    return data

target_dir = Path('/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_eval_target')
source_dir = Path('/content/drive/MyDrive/FreshCheck/artifacts/exp_c_plus_eval_source')

for eval_path in [
    target_dir / 'efficientnet_b0_eval.json',
    target_dir / 'swin_t_eval.json',
    source_dir / 'efficientnet_b0_eval.json',
    source_dir / 'swin_t_eval.json',
]:
    read_eval(eval_path)


In [ ]:
#@title 4) Prepare group-aware Kaggle train/val splits
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py prepare-splits --data-dir "$KAGGLE_DATA_DIR" --output-dir "$ARTIFACTS_DIR/splits"


In [ ]:
#@title 5) Train classifiers from the same CLI
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
EPOCHS = 15  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}
PRETRAINED = True  #@param {type:"boolean"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py train --train-csv "$ARTIFACTS_DIR/splits/kaggle_train.csv" --val-csv "$ARTIFACTS_DIR/splits/kaggle_val.csv" --output-dir "$ARTIFACTS_DIR/train" --models $MODELS --epochs $EPOCHS --batch-size $BATCH_SIZE --num-workers $NUM_WORKERS {'--pretrained' if PRETRAINED else '--no-pretrained'}


In [ ]:
#@title 6) Evaluate checkpoints on validation CSV
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
BATCH_SIZE = 16  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py evaluate --csv "$ARTIFACTS_DIR/splits/kaggle_val.csv" --checkpoint-dir "$ARTIFACTS_DIR/train/checkpoints" --output-dir "$ARTIFACTS_DIR/eval" --models $MODELS --batch-size $BATCH_SIZE --num-workers $NUM_WORKERS


In [ ]:
#@title 7) Predict on one image or one folder
INPUT_PATH = "/content/drive/MyDrive/FreshCheck/inference_samples"  #@param {type:"string"}
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
BATCH_SIZE = 16  #@param {type:"integer"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py predict --input-path "$INPUT_PATH" --checkpoint-dir "$ARTIFACTS_DIR/train/checkpoints" --output-dir "$ARTIFACTS_DIR/predict" --models $MODELS --batch-size $BATCH_SIZE


In [ ]:
#@title 8) Phase 2 foundation: CVAT XML -> GT masks + manifest CSV
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py prepare-cvat --thai-data-dir "$THAI_DATA_DIR" --cvat-xml-path "$CVAT_XML_PATH" --output-dir "$ARTIFACTS_DIR/phase2"


## Notes

- ถ้าแก้โค้ดใน GitHub แล้ว ให้กลับไปรัน cell `1) Clone / Update repository from GitHub`
- ไฟล์ model checkpoints จะถูกเก็บใน `MyDrive/FreshCheck/artifacts/train/checkpoints`
- ไฟล์ใหญ่ทั้งหมดควรอยู่บน Drive ไม่ควร commit ลง GitHub
- โมเดล gated (`SAM3`, `Florence-2`, `DINOv3`) ยังต้องเปิด access/token ก่อนค่อยทำ runner เพิ่ม
